In [ ]:
import os
from dvpio.read.image import read_czi
import spatialdata as sd
from spatialdata.models import Image2DModel

display_downsample = 8

czi_dir = r"/Users/rct905/Documents/MaxPlanck_HE_lung_skin/selected/"
czi_skin1a = r"skin1a.czi"
czi_lung2a = r"lung2a.czi"
czi_lung5a = r"lung5a.czi"

czi_path = os.path.join(czi_dir, czi_lung5a)

img = read_czi(czi_path, channels = 0)
image_model = Image2DModel.parse(img)
sdata = sd.SpatialData(
    images={"image": image_model}
)
type(sdata)

In [ ]:
img = sdata.images[list(sdata.images.keys())[0]]
img

In [ ]:
fullres = sdata.images["image"]

lowres = fullres.coarsen(
    {fullres.dims[-2]: display_downsample, fullres.dims[-1]: display_downsample},
    boundary="trim",
).mean()

sdata.images["image_vis"] = Image2DModel.parse(lowres)

import random
from shapely.geometry import box
import geopandas as gpd
from spatialdata.models import ShapesModel
import numpy as np

H = img.shape[1]  # image height
W = img.shape[2]  # image width
H = H // display_downsample
W = W // display_downsample

tile_size = 224 // display_downsample  # tile size in low-res coordinates
n_tiles = 20

tiles = []

# your sampling region
x_min, x_max = int(H * 0.3), int(H * 0.7)
y_min, y_max = int(W * 0.3), int(W * 0.7)

max_attempts = 10000
attempts = 0

while len(tiles) < n_tiles and attempts < max_attempts:
    attempts += 1

    x = random.randint(x_min, x_max - tile_size)
    y = random.randint(y_min, y_max - tile_size)

    new_tile = box(x, y, x + tile_size, y + tile_size)

    # check overlap
    if any(new_tile.intersects(existing) for existing in tiles):
        continue

    tiles.append(new_tile)

print(f"Generated {len(tiles)} non-overlapping tiles in {attempts} attempts")

tiles_gdf = gpd.GeoDataFrame(
    {"tile_id": [f"tile_{i}" for i in range(len(tiles))]},
    geometry=tiles,
)

sdata.shapes["tiles"] = ShapesModel.parse(tiles_gdf)

polygons = [
    np.array(g.exterior.coords)
    for g in tiles_gdf.geometry
    if g.geom_type == "Polygon"
]

In [ ]:
tiles_gdf

In [ ]:
from spatialdata.models import ShapesModel

sdata.shapes["tiles"] = ShapesModel.parse(tiles_gdf)

In [ ]:
sdata

In [ ]:
import napari

viewer = napari.Viewer()

viewer.add_image(
    sdata.images["image_vis"].data,
    channel_axis=0,
    name="WSI_preview",
)
viewer.add_shapes(
    polygons,
    shape_type="polygon",
    edge_color="red",
    face_color="red",
)
napari.run()

In [ ]:
from spatialdata.models import PointsModel
import numpy as np

points_layer = viewer.layers["calibration_points"]
image_points = points_layer.data
print(image_points)

sdata.points["calibration_points"] = PointsModel.parse(
    np.array(image_points)
)

In [ ]:
from spatialdata.models import ShapesModel
from shapely.geometry import Polygon
import geopandas as gpd
import numpy as np

square_layer = viewer.layers["polygons"]
image_square = square_layer.data
print(image_square)

polygons = [Polygon(coords) for coords in image_square]
square_gdf = gpd.GeoDataFrame(
    {"shape_id": [f"square_{i}" for i in range(len(polygons))]},
    geometry=polygons
)

sdata.shapes["square"] = ShapesModel.parse(square_gdf)

In [ ]:
pts = sdata.points["calibration_points"]
print(pts)

In [ ]:
sdata

In [ ]:
H = sdata.images["image"].data.shape[1]
print(H)

In [ ]:

from dvpio.write import write_lmd

path_lmd = os.path.join(czi_dir, "lung5a.xml")

affine_transformation = np.array([
    [1,  0, 0],
    [0, -1, H],
    [0,  0, 1]
])

write_lmd(
    path_lmd,
    sdata.shapes["tiles"],
    calibration_points=sdata.points["calibration_points"],
    affine_transformation=affine_transformation
)